# EXACT Phase-1 v40.5 One-Run Audit + Safe Upgrade

Runs the full audit/metrics/safe-upgrade pipeline in one pass. Attach `exact_eval_round1_Astatine.json` on Kaggle for full replay; otherwise it falls back to uploaded replay cases.


In [ ]:
from pathlib import Path
WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/mnt/data")
WORK.mkdir(parents=True, exist_ok=True)
SCRIPT = WORK / "exact_v40_5_one_run_pipeline.py"
SCRIPT.write_text('# -*- coding: utf-8 -*-\n"""\nEXACT Phase-1 v40.5 one-run audit + safe-upgrade pipeline.\n\nWhat it does in one command:\n1) Finds exact_eval_round1_Astatine.json when available.\n2) Replays the current abstain-safe v40.4/v40.5 MC certificate engine.\n3) Writes JSON for every stage: input audit, current pipeline, replay, gap audit,\n   metrics, overfit/underfit audit, safe-upgrade decision, and Claude handoff data.\n4) If full Phase-1 logs are absent, falls back to v40_4_phase1_replay_cases.json\n   and computes all metrics that are possible from the replay cases.\n\nDesign principle: never promote a wider rule unless it keeps wrong=[] and does not\nregress answer/premises correctness. YNU experimental support is audited but not\nblindly applied unless a full log verifies it safely.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport glob\nimport importlib.util\nimport json\nimport math\nimport os\nimport re\nimport sys\nfrom collections import Counter, defaultdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\n\nVERSION = "v40.5-one-run-safe-audit"\nMC_LABELS = ["A", "B", "C", "D"]\nYNU_LABELS = ["Yes", "No", "Uncertain"]\nALL_LABELS = MC_LABELS + YNU_LABELS\nABSTAIN = "__ABSTAIN__"\n\n# ---------------------------------------------------------------------------\n# v40.4/v40.5 conservative entity-grounded conjunctive Horn engine\n# ---------------------------------------------------------------------------\nSTOP = {\n    \'a\',\'an\',\'the\',\'of\',\'to\',\'in\',\'on\',\'at\',\'for\',\'and\',\'or\',\'that\',\'this\',\'their\',\'its\',\'it\',\'they\',\'them\',\n    \'is\',\'are\',\'was\',\'were\',\'be\',\'been\',\'has\',\'have\',\'had\',\'then\',\'if\',\'no\',\'not\',\'with\',\'as\',\'by\',\'from\',\n    \'artifact\',\'package\',\'manuscript\',\'sample\',\'batch\',\'item\',\'device\',\'record\',\'file\',\'student\',\'case\',\n    \'premise\',\'premises\',\'according\',\'conclusion\',\'logically\',\'supported\',\'correct\',\'statement\',\'based\'\n}\n\ndef _stem(t: str) -> str:\n    if re.search(r\'(ss|us|is)$\', t):\n        return t\n    if re.search(r\'(ches|shes|xes|zes|ses)$\', t):\n        return t[:-2]\n    if re.search(r\'ies$\', t):\n        return t[:-3] + \'y\'\n    if t.endswith(\'s\'):\n        t = t[:-1]\n    return re.sub(r\'(ing|ed)$\', \'\', t)\n\ndef atom_key(phrase: Any) -> Optional[str]:\n    s = re.sub(r\'(?<!^)(?=[A-Z])\', \' \', str(phrase)).lower()\n    nums = re.findall(r\'\\d+\', s)\n    toks = [_stem(w) for w in re.findall(r\'[a-zA-Z]+\', s)]\n    toks = [t for t in toks if t and t not in STOP and len(t) > 2]\n    keys = sorted(set(toks)) + ["N" + n for n in sorted(set(nums))]\n    return "".join(w.capitalize() for w in keys) if keys else None\n\n_LEAD = re.compile(r"^\\s*(if|then|that|who|which|it|its|their|this)\\b", re.I)\n_VERB = re.compile(\n    r"\\b(cannot|can not|can|could|may|might|must|should|shall|will|would|"\n    r"is not|are not|was not|were not|isn\'t|aren\'t|is|are|was|were|"\n    r"has no|have no|had no|has|have|had|lacks?|without|"\n    r"requires?|needs?|contains?|completed?|enters?|gains?|receives?|provides?|"\n    r"shows?|states?|holds?|carries|monitors?|captures?|eligible|allowed|approved|"\n    r"assigned|listed|qualifies?|qualified|searchable|recommended|administered|"\n    r"displayed|dispatch(?:ed)?|review(?:ed)?|released?|closed|"\n    r"be|been|being)\\b", re.I\n)\nACTION_VERBS = {\n    \'receives\',\'receive\',\'provides\',\'provide\',\'shows\',\'show\',\'states\',\'state\',\'monitors\',\'monitor\',\'captures\',\'capture\',\n    \'enters\',\'enter\',\'requires\',\'require\',\'needs\',\'need\',\'gains\',\'gain\',\'completed\',\'complete\',\'contains\',\'contain\',\n    \'reports\',\'report\',\'releases\',\'release\',\'passes\',\'pass\',\'improves\',\'improve\',\'supports\',\'support\',\'recommends\',\n    \'recommend\',\'administered\',\'administer\',\'approved\',\'approve\',\'listed\',\'list\',\'qualifies\',\'qualify\',\'qualified\',\n    \'searchable\',\'displayed\',\'display\',\'dispatch\',\'dispatched\',\'review\',\'reviewed\',\'released\',\'release\',\'closed\',\'close\'\n}\n_NEG_RE = re.compile(\n    r"\\b(no|not|cannot|can not|never|lacks?|without|isn\'t|aren\'t|incomplete|missing|lacking|nor|"\n    r"un(?:able|verified|established|approved|cleared|safe|eligible))\\b", re.I\n)\n\ndef to_literal(clause: Any) -> Optional[Tuple[str, bool]]:\n    c = str(clause).strip().rstrip(\'.?\').strip()\n    c = _LEAD.sub(\'\', c).strip()\n    # normalize common question fragments into declarative-ish clauses\n    c = re.sub(r"^does\\s+(.+?)\\s+have\\s+(.+)$", r"\\1 has \\2", c, flags=re.I)\n    c = re.sub(r"^do\\s+the\\s+premises\\s+(?:prove|establish|show|guarantee)\\s+that\\s+", "", c, flags=re.I)\n    neg = bool(_NEG_RE.search(c))\n    m = _VERB.search(c)\n    pred = c[m.end():].strip() if m else c\n    verb = (m.group(1).lower() if m else \'\')\n    if m and verb in ACTION_VERBS:\n        pred = verb + \' \' + pred\n    for _ in range(5):\n        pred = re.sub(r"^\\s*(be|been|being|to|a|an|the|no|not|its|their|for|by)\\b", "", pred, flags=re.I).strip()\n    a = atom_key(pred)\n    return (a, neg) if a else None\n\ndef parse_premise(p: Any) -> Optional[Tuple]:\n    s = str(p).strip()\n    m = re.search(r\'^\\s*if\\b(.+?),?\\s*\\bthen\\b(.+)$\', s, re.I)\n    if m:\n        ante = re.split(r\'\\band\\b\', m.group(1), flags=re.I)\n        lits = [to_literal(x) for x in ante]\n        lits = [l for l in lits if l]\n        con = to_literal(m.group(2))\n        if con and lits:\n            return (\'rule\', lits, con)\n        return None\n    m2 = re.search(\n        r\'^\\s*(every|all)\\s+[a-zA-Z]+s?\\s+(.+?)\\s+\\b(can|may|must|should|will|would|receives?|gets?|gains?|provides?|captures?|monitors?|requires?|needs?|is|are|qualifies?|eligible|listed|recommended)\\b\\s+(.+)$\',\n        s, re.I,\n    )\n    if m2:\n        cond = m2.group(2).strip()\n        cons = (m2.group(3) + " " + m2.group(4)).strip()\n        litc = to_literal(cond)\n        litd = to_literal(cons)\n        if litc and litd:\n            return (\'rule\', [litc], litd)\n    if re.search(r\'^\\s*(no premise|it (is|cannot)|unknown|there is no information)\', s, re.I):\n        return None\n    lit = to_literal(s)\n    return (\'fact\', lit) if lit else None\n\ndef solve_entity(premises: Sequence[Any]) -> Dict[str, Tuple[bool, List[int]]]:\n    facts: Dict[str, Tuple[bool, List[int]]] = {}\n    rules = []\n    for i, p in enumerate(premises):\n        pp = parse_premise(p)\n        if not pp:\n            continue\n        if pp[0] == \'fact\':\n            a, neg = pp[1]\n            facts.setdefault(a, (not neg, [i]))\n        else:\n            rules.append((i, pp[1], pp[2]))\n    changed = True\n    while changed:\n        changed = False\n        for i, lits, con in rules:\n            ca, cneg = con\n            ok = True\n            path = [i]\n            for a, neg in lits:\n                if a in facts and facts[a][0] == (not neg):\n                    path += facts[a][1]\n                else:\n                    ok = False\n                    break\n            if ok and ca not in facts:\n                facts[ca] = ((not cneg), sorted(set(path)))\n                changed = True\n    return facts\n\n_META_RE = re.compile(\n    r"\\b(not (?:yet )?(?:established|confirmed|verified|approved|cleared|determined)|"\n    r"cannot be (?:established|confirmed)|unsupported|is not established|no premise|undetermined|not (?:available|present))\\b",\n    re.I,\n)\n\ndef decompose_option(opt: Any) -> List[Tuple[Optional[Tuple[str, bool]], bool, str]]:\n    t = re.sub(r\'^\\s*[A-Da-d][.):]\\s*\', \'\', str(opt)).strip()\n    t = re.split(r\'\\bbecause\\b\', t, maxsplit=1, flags=re.I)[0].strip()\n    parts = re.split(r\',\\s*but\\s+|\\s+but\\s+|;\\s+|\\s+while\\s+|\\s+whereas\\s+|\\s+and\\s+\', t, flags=re.I)\n    claims = []\n    for p in parts:\n        p = p.strip()\n        if not p:\n            continue\n        is_meta = bool(_META_RE.search(p))\n        lit = to_literal(p)\n        claims.append((lit, is_meta, p))\n    return claims\n\ndef answer_mc(premises: Sequence[Any], options: Sequence[Any]) -> Tuple[Optional[str], List[int], str, Dict[str, Tuple[str, List[int]]]]:\n    facts = solve_entity(premises)\n    res: Dict[str, Tuple[str, List[int]]] = {}\n    for lab, opt in zip("ABCD", options):\n        claims = decompose_option(opt)\n        if not claims:\n            res[lab] = (\'UNSUP\', [])\n            continue\n        status = \'PROVEN\'\n        path: List[int] = []\n        for lit, is_meta, txt in claims:\n            if lit is None:\n                status = \'UNSUP\'\n                break\n            a, neg = lit\n            have = a in facts\n            val = facts[a][0] if have else None\n            if is_meta:\n                if have and val is True:\n                    status = \'DISPROVEN\'\n                    break\n            else:\n                if have and val == (not neg):\n                    path += facts[a][1]\n                elif have and val == neg:\n                    status = \'DISPROVEN\'\n                    break\n                else:\n                    status = \'UNSUP\'\n                    break\n        res[lab] = (status, sorted(set(path)))\n    proven = [l for l in res if res[l][0] == \'PROVEN\']\n    if len(proven) == 1:\n        return proven[0], res[proven[0]][1], \'entity_unique_proof\', res\n    return None, [], (\'multiple\' if proven else \'none\'), res\n\n# Experimental target parser for YNU. It is audited but not promoted unless verified.\n_PROOF_NEGATIVE_Q = re.compile(r"\\b(premises\\s+(?:prove|establish|show)|establish|prove|guarantee|satisfy every requirement)\\b", re.I)\n\ndef question_to_claim(query: Any) -> Tuple[Optional[Tuple[str, bool]], str]:\n    q = str(query).strip()\n    q = re.sub(r"\\s+", " ", q).strip().rstrip(\'?\')\n    q = re.sub(r",\\s*according to the premises", "", q, flags=re.I)\n    q = re.sub(r"\\s+according to the premises", "", q, flags=re.I)\n    mode = "direct_unknown_absence"\n    if _PROOF_NEGATIVE_Q.search(q):\n        mode = "proof_absence_means_no"\n    # Peel question shells.\n    patterns = [\n        r"^do\\s+the\\s+premises\\s+(?:prove|establish|show|guarantee)\\s+that\\s+(.+)$",\n        r"^is\\s+(.+)$",\n        r"^are\\s+(.+)$",\n        r"^does\\s+(.+)$",\n        r"^do\\s+(.+)$",\n        r"^can\\s+(.+)$",\n        r"^may\\s+(.+)$",\n        r"^must\\s+(.+)$",\n        r"^should\\s+(.+)$",\n    ]\n    claim = None\n    for pat in patterns:\n        m = re.search(pat, q, flags=re.I)\n        if m:\n            claim = m.group(1).strip()\n            break\n    if claim is None:\n        claim = q\n    # Convert simple auxiliaries.\n    claim = re.sub(r"^(.+?)\\s+have\\s+(.+)$", r"\\1 has \\2", claim, flags=re.I)\n    lit = to_literal(claim)\n    return lit, mode\n\ndef answer_ynu_experimental(premises: Sequence[Any], query: Any) -> Tuple[Optional[str], List[int], str, Dict[str, Any]]:\n    lit, mode = question_to_claim(query)\n    facts = solve_entity(premises)\n    debug = {"target_literal": lit, "mode": mode}\n    if lit is None:\n        return None, [], "ynu_no_target", debug\n    a, neg = lit\n    if a in facts:\n        val, path = facts[a]\n        if val == (not neg):\n            return "Yes", path, "ynu_target_proven", debug\n        if val == neg:\n            return "No", path, "ynu_target_disproven", debug\n    # Conservative: only proof-form questions may map absence of proof to No.\n    # Direct property questions stay Uncertain only when a known uncertainty cue exists in premises.\n    if mode == "proof_absence_means_no":\n        return "No", [], "ynu_proof_absence_means_no_no_certificate", debug\n    return None, [], "ynu_direct_absence_abstain", debug\n\ndef opt_texts(rp: Dict[str, Any]) -> List[str]:\n    query = rp.get("query", "") or ""\n    found = re.findall(r"(?:^|\\n)\\s*([A-D])[.)]\\s*(.+?)(?=\\n\\s*[A-D][.)]|\\Z)", query, flags=re.S)\n    f = [text.strip().replace("\\n", " ") for _, text in found]\n    return f if len(f) >= 2 else (rp.get("options") or [])\n\ndef classify_task(expected_answer: Any, query: str = "", options: Optional[Sequence[Any]] = None) -> str:\n    ea = str(expected_answer).strip()\n    if ea.upper() in MC_LABELS:\n        return "MC"\n    if ea in YNU_LABELS:\n        return "YNU"\n    opts = options or []\n    if len(opts) >= 2:\n        return "MC"\n    if re.match(r"^(is|are|do|does|can|may|must|should)\\b", str(query).strip(), flags=re.I):\n        return "YNU"\n    return "OTHER"\n\n# ---------------------------------------------------------------------------\n# Data loading and replay\n# ---------------------------------------------------------------------------\ndef find_phase1(explicit: Optional[str] = None) -> Optional[str]:\n    if explicit and os.path.exists(explicit):\n        return explicit\n    patterns = [\n        "exact_eval_round1_Astatine.json",\n        "./exact_eval_round1_Astatine.json",\n        "/kaggle/working/**/exact_eval_round1_Astatine.json",\n        "/kaggle/input/**/exact_eval_round1_Astatine.json",\n        "/mnt/data/**/exact_eval_round1_Astatine.json",\n    ]\n    hits: List[str] = []\n    for pat in patterns:\n        if any(ch in pat for ch in "*?"):\n            hits.extend(glob.glob(pat, recursive=True))\n        elif os.path.exists(pat):\n            hits.append(pat)\n    hits = sorted(set(hits))\n    return hits[0] if hits else None\n\ndef load_phase1_logs(path: str) -> List[Dict[str, Any]]:\n    data = json.load(open(path, encoding="utf-8"))\n    logs = [l for l in data.get("logs", []) if l.get("type") == "type1"]\n    return logs\n\ndef run_variant_on_logs(logs: Sequence[Dict[str, Any]], variant: str) -> List[Dict[str, Any]]:\n    rows = []\n    for l in logs:\n        rp = l.get("request_payload", {}) or {}\n        exp = l.get("expected", {}) or {}\n        expected_answer = exp.get("answer")\n        task = classify_task(expected_answer, rp.get("query", ""), rp.get("options"))\n        premises = rp.get("premises", []) or []\n        options = opt_texts(rp)\n        a: Optional[str] = None\n        pu: List[int] = []\n        why = "not_run"\n        res: Dict[str, Any] = {}\n        if task == "MC":\n            a, pu, why, res = answer_mc(premises, options)\n        elif task == "YNU" and variant == "mc_plus_ynu_experimental":\n            a, pu, why, res = answer_ynu_experimental(premises, rp.get("query", ""))\n        else:\n            a, pu, why, res = None, [], "task_abstain", {}\n        ea_norm = str(expected_answer or "").strip()\n        ans_ok = a is not None and str(a).strip().upper() == ea_norm.upper()\n        prem_ok = a is not None and sorted(pu) == sorted(exp.get("premises_used") or [])\n        rows.append({\n            "query_id": l.get("query_id"),\n            "task": task,\n            "old_status": l.get("status"),\n            "expected_answer": expected_answer,\n            "expected_premises_used": exp.get("premises_used"),\n            "v_answer": a,\n            "v_premises_used": pu,\n            "rule": why,\n            "answer_ok": ans_ok,\n            "premises_ok": prem_ok,\n            "query": str(rp.get("query", ""))[:500],\n            "option_status": res,\n        })\n    return rows\n\n# ---------------------------------------------------------------------------\n# Metrics\n# ---------------------------------------------------------------------------\ndef safe_div(a: float, b: float) -> Optional[float]:\n    return None if b == 0 else a / b\n\ndef round_float(x: Any, nd: int = 6) -> Any:\n    if isinstance(x, float):\n        return round(x, nd)\n    return x\n\ndef classification_metrics(rows: Sequence[Dict[str, Any]], labels: Sequence[str], pred_key: str = "v_answer") -> Dict[str, Any]:\n    n = len(rows)\n    correct = 0\n    fired = 0\n    per = {}\n    for lab in labels:\n        tp = fp = fn = 0\n        support = 0\n        for r in rows:\n            y = str(r.get("expected_answer", "")).strip()\n            p = r.get(pred_key)\n            p = str(p).strip() if p is not None else ABSTAIN\n            if y == lab:\n                support += 1\n            if p != ABSTAIN:\n                fired += 0  # counted once below\n            if p == lab and y == lab:\n                tp += 1\n            elif p == lab and y != lab:\n                fp += 1\n            elif p != lab and y == lab:\n                fn += 1\n        precision = safe_div(tp, tp + fp)\n        recall = safe_div(tp, tp + fn)\n        f1 = None if precision is None or recall is None or (precision + recall) == 0 else 2 * precision * recall / (precision + recall)\n        per[lab] = {\n            "support": support,\n            "tp": tp,\n            "fp": fp,\n            "fn": fn,\n            "precision": round_float(precision) if precision is not None else None,\n            "recall": round_float(recall) if recall is not None else None,\n            "f1": round_float(f1) if f1 is not None else None,\n        }\n    fired = sum(1 for r in rows if r.get(pred_key) is not None)\n    correct = sum(1 for r in rows if str(r.get(pred_key, "")).strip().upper() == str(r.get("expected_answer", "")).strip().upper())\n    # Macro treats undefined precision/F1 as 0 for strict abstain-as-miss scoring.\n    macro_precision = sum((per[l]["precision"] or 0.0) for l in labels) / len(labels) if labels else None\n    macro_recall = sum((per[l]["recall"] or 0.0) for l in labels) / len(labels) if labels else None\n    macro_f1 = sum((per[l]["f1"] or 0.0) for l in labels) / len(labels) if labels else None\n    return {\n        "n": n,\n        "fired": fired,\n        "abstained": n - fired,\n        "coverage": round_float(safe_div(fired, n) or 0.0),\n        "strict_accuracy_abstain_as_wrong": round_float(safe_div(correct, n) or 0.0),\n        "fired_accuracy": round_float(safe_div(correct, fired)) if fired else None,\n        "macro_precision_strict": round_float(macro_precision),\n        "macro_recall_strict": round_float(macro_recall),\n        "macro_f1_strict": round_float(macro_f1),\n        "per_label": per,\n    }\n\ndef baseline_status_metrics(rows: Sequence[Dict[str, Any]]) -> Dict[str, Any]:\n    n = len(rows)\n    c = Counter(r.get("old_status") for r in rows)\n    exact_correct = c.get("correct", 0)\n    answer_correct = c.get("correct", 0) + c.get("wrong_premises_used", 0)\n    by_label = defaultdict(lambda: Counter())\n    for r in rows:\n        by_label[str(r.get("expected_answer"))][r.get("old_status")] += 1\n    label_recall = {}\n    for lab, cnt in by_label.items():\n        support = sum(cnt.values())\n        label_recall[lab] = {\n            "support": support,\n            "old_exact_recall_by_status": round_float(safe_div(cnt.get("correct", 0), support) or 0.0),\n            "old_answer_recall_by_status": round_float(safe_div(cnt.get("correct", 0) + cnt.get("wrong_premises_used", 0), support) or 0.0),\n            "status_counts": dict(cnt),\n        }\n    return {\n        "n": n,\n        "status_counts": dict(c),\n        "old_exact_accuracy_status_correct_only": round_float(safe_div(exact_correct, n) or 0.0),\n        "old_answer_accuracy_status_correct_or_wrong_premises": round_float(safe_div(answer_correct, n) or 0.0),\n        "note": "F1/precision for the old model require raw old predicted labels; status-only cases cannot reconstruct them.",\n        "per_expected_label_status_recall": label_recall,\n    }\n\ndef apply_override_status(rows: Sequence[Dict[str, Any]], pred_key: str = "v_answer") -> List[Dict[str, Any]]:\n    out = []\n    for r in rows:\n        x = copy.deepcopy(r)\n        old = r.get("old_status")\n        if r.get(pred_key) is None:\n            x["applied_status"] = old\n        else:\n            if r.get("answer_ok") and r.get("premises_ok"):\n                x["applied_status"] = "correct"\n            elif r.get("answer_ok"):\n                x["applied_status"] = "wrong_premises_used"\n            else:\n                x["applied_status"] = "wrong_answer"\n        out.append(x)\n    return out\n\ndef applied_status_metrics(rows: Sequence[Dict[str, Any]]) -> Dict[str, Any]:\n    n = len(rows)\n    c = Counter(r.get("applied_status") for r in rows)\n    exact_correct = c.get("correct", 0)\n    answer_correct = c.get("correct", 0) + c.get("wrong_premises_used", 0)\n    return {\n        "n": n,\n        "status_counts_after_safe_override": dict(c),\n        "estimated_exact_accuracy_after_override": round_float(safe_div(exact_correct, n) or 0.0),\n        "estimated_answer_accuracy_after_override": round_float(safe_div(answer_correct, n) or 0.0),\n        "note": "This estimate is exact for status categories if old_status semantics are correct. Label F1 still needs raw old predictions for abstained rows.",\n    }\n\ndef split_rows(rows: Sequence[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:\n    return {\n        "ALL": list(rows),\n        "MC": [r for r in rows if classify_task(r.get("expected_answer"), r.get("query", "")) == "MC" or r.get("task") == "MC"],\n        "YNU": [r for r in rows if classify_task(r.get("expected_answer"), r.get("query", "")) == "YNU" or r.get("task") == "YNU"],\n    }\n\ndef full_metrics(rows: Sequence[Dict[str, Any]], source_note: str) -> Dict[str, Any]:\n    groups = split_rows(rows)\n    out = {"source_note": source_note, "groups": {}}\n    for name, rs in groups.items():\n        labels = MC_LABELS if name == "MC" else YNU_LABELS if name == "YNU" else ALL_LABELS\n        applied = apply_override_status(rs)\n        out["groups"][name] = {\n            "baseline_status_metrics": baseline_status_metrics(rs),\n            "certificate_engine_metrics": classification_metrics(rs, labels),\n            "after_safe_override_status_metrics": applied_status_metrics(applied),\n        }\n    return out\n\ndef replay_report_from_rows(rows: Sequence[Dict[str, Any]]) -> Dict[str, Any]:\n    fired_rows = [r for r in rows if r.get("v_answer") is not None or r.get("v40_answer") is not None]\n    # support both raw v40 cases and new rows\n    def get_answer(r): return r.get("v_answer", r.get("v40_answer"))\n    def get_pu(r): return r.get("v_premises_used", r.get("v40_premises_used", []))\n    wrong = []\n    fixed = []\n    aok = pok = 0\n    for r in rows:\n        a = get_answer(r)\n        if a is None:\n            continue\n        ok = str(a).strip().upper() == str(r.get("expected_answer", "")).strip().upper()\n        prem_ok = sorted(get_pu(r)) == sorted(r.get("expected_premises_used") or [])\n        aok += int(ok)\n        pok += int(prem_ok)\n        if ok and r.get("old_status") != "correct":\n            fixed.append(r.get("query_id"))\n        if not ok:\n            wrong.append({"query_id": r.get("query_id"), "exp": r.get("expected_answer"), "got": a})\n    return {\n        "n": len(rows),\n        "fired": len(fired_rows),\n        "answer_correct": aok,\n        "premises_used_correct": pok,\n        "wrong": wrong,\n        "fixed_old_wrong": fixed,\n        "abstained": len(rows) - len(fired_rows),\n        "precision_when_fired": round_float(safe_div(aok, len(fired_rows))) if fired_rows else None,\n        "coverage": round_float(safe_div(len(fired_rows), len(rows)) or 0.0),\n        "gate": "ABSTAIN_SAFE" if not wrong else "HAS_WRONG_FIX_BEFORE_APPLY",\n    }\n\n# ---------------------------------------------------------------------------\n# JSON/report writing\n# ---------------------------------------------------------------------------\ndef write_json(path: Path, obj: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")\n\ndef write_text(path: Path, text: str) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(text, encoding="utf-8")\n\ndef now_iso() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\ndef normalize_uploaded_cases(cases: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    rows = []\n    for r in cases:\n        x = dict(r)\n        x["task"] = classify_task(x.get("expected_answer"), x.get("query", ""))\n        x["v_answer"] = x.get("v40_answer")\n        x["v_premises_used"] = x.get("v40_premises_used", [])\n        rows.append(x)\n    return rows\n\ndef make_gap_audit(rows: Sequence[Dict[str, Any]]) -> Dict[str, Any]:\n    groups = split_rows(rows)\n    out = {}\n    for name, rs in groups.items():\n        abst = [r for r in rs if r.get("v_answer") is None and r.get("v40_answer") is None]\n        fired = [r for r in rs if r.get("v_answer") is not None or r.get("v40_answer") is not None]\n        out[name] = {\n            "n": len(rs),\n            "fired": len(fired),\n            "abstained": len(abst),\n            "abstain_rate": round_float(safe_div(len(abst), len(rs)) or 0.0),\n            "old_status_counts_in_abstains": dict(Counter(r.get("old_status") for r in abst)),\n            "expected_answer_counts_in_abstains": dict(Counter(str(r.get("expected_answer")) for r in abst)),\n            "top_abstain_examples": [\n                {"query_id": r.get("query_id"), "expected_answer": r.get("expected_answer"), "old_status": r.get("old_status"), "rule": r.get("rule"), "query": r.get("query", "")[:180]}\n                for r in abst[:12]\n            ],\n        }\n    return out\n\ndef make_current_pipeline_json(source_mode: str) -> Dict[str, Any]:\n    return {\n        "version": VERSION,\n        "source_mode": source_mode,\n        "pipeline": [\n            {"stage": 0, "name": "input_audit", "description": "Find full Phase-1 log if available; otherwise use uploaded replay cases."},\n            {"stage": 1, "name": "parse_task", "description": "Split Type1 into MC labels A-D and YNU labels Yes/No/Uncertain."},\n            {"stage": 2, "name": "entity_horn_solver", "description": "Parse facts/rules into single-entity literals; forward-chain Horn implications; retain certificate premise indices."},\n            {"stage": 3, "name": "mc_certificate_answer", "description": "For A-D options, prove/decompose claims; fire only on exactly one PROVEN option."},\n            {"stage": 4, "name": "ynu_experimental_audit", "description": "YNU target parser is audited only; not promoted unless full-log gate has zero wrong and no premises regression."},\n            {"stage": 5, "name": "safe_override", "description": "Override existing model only when certificate answer is non-null and gate passes; otherwise keep original model output."},\n            {"stage": 6, "name": "metrics_and_gap_audit", "description": "Write coverage, fired precision, status accuracy, F1/recall where reconstructable, abstain breakdown, overfit/underfit risk."},\n        ],\n        "hard_gate": {\n            "must_have_wrong_empty": True,\n            "must_have_all_fired_answers_correct": True,\n            "must_not_regress_premises_used_correct": True,\n            "on_fail": "reject_candidate_and_keep_previous_stage",\n        },\n    }\n\ndef make_overfit_underfit_audit(rows: Sequence[Dict[str, Any]], source_mode: str) -> Dict[str, Any]:\n    gap = make_gap_audit(rows)\n    report = replay_report_from_rows(rows)\n    mc_n = gap["MC"]["n"]\n    ynu_n = gap["YNU"]["n"]\n    return {\n        "version": VERSION,\n        "source_mode": source_mode,\n        "summary": {\n            "overfit_status": "inconclusive_from_replay_only" if source_mode != "phase1_log" else "needs_holdout_or_cross_log_check",\n            "underfit_status": "high_coverage_underfit",\n            "why": [\n                "The rule layer is not trained statistically, so classic train-loss/test-loss overfit cannot be measured from this artifact alone.",\n                "Overfit risk appears when rules are hand-tuned to the same 25 Phase-1 cases; use staged zero-wrong promotion and a separate holdout/full log.",\n                f"Underfit/low-coverage is visible: fired={report[\'fired\']}/{report[\'n\']} and abstained={report[\'abstained\']}/{report[\'n\']}.",\n                f"MC abstains={gap[\'MC\'][\'abstained\']}/{mc_n}; YNU abstains={gap[\'YNU\'][\'abstained\']}/{ynu_n}.",\n            ],\n        },\n        "risk_scores_qualitative": {\n            "current_v40_4_overfit_risk": "low_to_medium_because_rules_are_broad_but_validated_on_small_n",\n            "future_parser_expansion_overfit_risk": "medium_to_high_unless_checked_on_full_log_or_holdout",\n            "current_underfit_risk": "high_because_coverage_is_low_and_YNU_is_not_covered",\n        },\n        "controls": [\n            "Do not apply YNU experimental answers unless full-log wrong=[] and fired premises certificates are correct or explicitly accepted.",\n            "Keep T1_0031-style multiple-proven cases as abstain, never force-tie-break.",\n            "Track every promoted rule in a stage JSON with fired/wrong/fixed/abstained counts.",\n        ],\n    }\n\ndef make_stage_decision(stage_reports: List[Dict[str, Any]]) -> Dict[str, Any]:\n    accepted = []\n    rejected = []\n    for s in stage_reports:\n        rep = s.get("replay_report") or {}\n        if s.get("runnable") is False:\n            rejected.append({"stage": s["stage"], "reason": s.get("reason", "not runnable")})\n            continue\n        if rep.get("wrong") == [] and rep.get("answer_correct") == rep.get("fired"):\n            accepted.append({"stage": s["stage"], "name": s["name"], "gate": "accepted_for_audit_or_safe_apply"})\n        else:\n            rejected.append({"stage": s["stage"], "name": s["name"], "reason": "gate_failed_or_unverified"})\n    return {"accepted": accepted, "rejected_or_not_promoted": rejected}\n\ndef make_claude_handoff_text(metrics: Dict[str, Any], gap: Dict[str, Any], source_mode: str) -> str:\n    all_m = metrics["groups"]["ALL"]\n    mc_m = metrics["groups"]["MC"]\n    ynu_m = metrics["groups"]["YNU"]\n    return f"""# Claude handoff — EXACT Phase-1 v40.5 one-run audit\n\n## Context\nWe are hardening an abstain-safe symbolic certificate layer for EXACT Phase-1 Type1. Current engine is an entity-grounded conjunctive Horn forward-chain solver. It should override the model only when it has a unique proof certificate.\n\nSource mode for this run: `{source_mode}`.\n\n## Current result summary\n- All cases n: {all_m[\'certificate_engine_metrics\'][\'n\']}\n- Fired: {all_m[\'certificate_engine_metrics\'][\'fired\']}\n- Abstained: {all_m[\'certificate_engine_metrics\'][\'abstained\']}\n- Fired accuracy: {all_m[\'certificate_engine_metrics\'][\'fired_accuracy\']}\n- Strict accuracy if abstain is wrong: {all_m[\'certificate_engine_metrics\'][\'strict_accuracy_abstain_as_wrong\']}\n- MC fired/total: {mc_m[\'certificate_engine_metrics\'][\'fired\']}/{mc_m[\'certificate_engine_metrics\'][\'n\']}\n- YNU fired/total: {ynu_m[\'certificate_engine_metrics\'][\'fired\']}/{ynu_m[\'certificate_engine_metrics\'][\'n\']}\n\n## Important verdict\nDo NOT broaden by heuristic tie-break. Keep T1_0031-like `multiple` cases as abstain. Current value is precision/certificate safety, not coverage.\n\n## Underfit / overfit\n- Underfit is high because coverage is low: ALL abstains={gap[\'ALL\'][\'abstained\']}/{gap[\'ALL\'][\'n\']}, MC abstains={gap[\'MC\'][\'abstained\']}/{gap[\'MC\'][\'n\']}, YNU abstains={gap[\'YNU\'][\'abstained\']}/{gap[\'YNU\'][\'n\']}.\n- Classic overfit cannot be measured from replay cases alone. Risk becomes high only if new parser rules are tuned directly to the same 25 cases without a full-log/holdout gate.\n\n## Requested next action for Claude\nReview `exact_v40_5_one_run_pipeline.py` and the stage JSON files. Improve only by adding certificate-preserving parser rules. Any candidate must output stage JSON and pass:\n\n```python\nwrong == []\nanswer_correct == fired\npremises_used_correct does not regress\n```\n\nNever promote a YNU heuristic unless the full `exact_eval_round1_Astatine.json` verifies it. If full log is unavailable, keep YNU as audit-only.\n"""\n\ndef main(argv: Optional[Sequence[str]] = None) -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--phase1", default=None, help="Path to exact_eval_round1_Astatine.json")\n    ap.add_argument("--cases", default=None, help="Fallback path to v40_4_phase1_replay_cases.json")\n    ap.add_argument("--out", default=None, help="Output directory")\n    args = ap.parse_args(argv)\n\n    default_root = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/mnt/data")\n    out = Path(args.out) if args.out else default_root / "exact_v40_5_outputs"\n    out.mkdir(parents=True, exist_ok=True)\n\n    phase1_path = find_phase1(args.phase1)\n    source_mode = "phase1_log" if phase1_path else "replay_cases_only"\n    input_audit = {\n        "version": VERSION,\n        "created_utc": now_iso(),\n        "phase1_path": phase1_path,\n        "source_mode": source_mode,\n        "out_dir": str(out),\n        "warning": None,\n    }\n\n    stage_reports: List[Dict[str, Any]] = []\n\n    if phase1_path:\n        logs = load_phase1_logs(phase1_path)\n        input_audit["n_type1_logs"] = len(logs)\n        # Stage 1: MC-only conservative.\n        rows_mc = run_variant_on_logs(logs, "mc_only")\n        rep_mc = replay_report_from_rows(rows_mc)\n        stage_reports.append({"stage": 1, "name": "v40_5_mc_only_certificate", "runnable": True, "replay_report": rep_mc})\n        write_json(out / "stage_01_v40_5_mc_only_cases.json", rows_mc)\n        # Stage 2: YNU experimental audit, not automatic apply unless gate passes.\n        rows_ynu = run_variant_on_logs(logs, "mc_plus_ynu_experimental")\n        rep_ynu = replay_report_from_rows(rows_ynu)\n        stage_reports.append({"stage": 2, "name": "v40_5_mc_plus_ynu_experimental_AUDIT_ONLY", "runnable": True, "replay_report": rep_ynu, "promotion_policy": "only_if_zero_wrong_and_premises_ok; otherwise reject"})\n        write_json(out / "stage_02_v40_5_mc_plus_ynu_experimental_cases.json", rows_ynu)\n        # Safe selected rows: currently choose MC-only unless experimental strictly dominates safely.\n        selected_rows = rows_mc\n        selected_name = "v40_5_mc_only_certificate"\n        if rep_ynu.get("wrong") == [] and rep_ynu.get("answer_correct") == rep_ynu.get("fired") and rep_ynu.get("fired", 0) >= rep_mc.get("fired", 0):\n            # Still conservative about premises: only promote if all fired premise certificates match exactly.\n            if rep_ynu.get("premises_used_correct") == rep_ynu.get("fired"):\n                selected_rows = rows_ynu\n                selected_name = "v40_5_mc_plus_ynu_experimental_PROMOTED"\n        write_json(out / "stage_03_selected_cases.json", selected_rows)\n    else:\n        cases_path = args.cases\n        if not cases_path:\n            for p in ["/mnt/data/v40_4_phase1_replay_cases.json", "./v40_4_phase1_replay_cases.json", "/kaggle/working/v40_4_phase1_replay_cases.json"]:\n                if os.path.exists(p):\n                    cases_path = p\n                    break\n        if not cases_path or not os.path.exists(cases_path):\n            input_audit["warning"] = "No full phase1 log and no replay cases found. Only pipeline manifest will be written."\n            rows = []\n        else:\n            raw_cases = json.load(open(cases_path, encoding="utf-8"))\n            rows = normalize_uploaded_cases(raw_cases)\n            input_audit["cases_path"] = cases_path\n            input_audit["n_replay_cases"] = len(rows)\n        selected_rows = rows\n        selected_name = "uploaded_v40_4_replay_cases_no_full_premises"\n        stage_reports.append({\n            "stage": 1,\n            "name": "uploaded_v40_4_replay_cases_audit",\n            "runnable": bool(rows),\n            "reason": None if rows else "missing_cases",\n            "replay_report": replay_report_from_rows(rows) if rows else {},\n        })\n        stage_reports.append({\n            "stage": 2,\n            "name": "v40_5_full_log_upgrade_candidates",\n            "runnable": False,\n            "reason": "Full exact_eval_round1_Astatine.json is required to rerun parser upgrades against premises.",\n        })\n\n    # Common reports.\n    current_pipeline = make_current_pipeline_json(source_mode)\n    replay_report = replay_report_from_rows(selected_rows)\n    metrics = full_metrics(selected_rows, source_mode)\n    gap = make_gap_audit(selected_rows)\n    overfit_underfit = make_overfit_underfit_audit(selected_rows, source_mode)\n    decision = make_stage_decision(stage_reports)\n    manifest = {\n        "version": VERSION,\n        "created_utc": now_iso(),\n        "source_mode": source_mode,\n        "selected_stage": selected_name,\n        "files": {},\n    }\n\n    write_json(out / "stage_00_input_audit.json", input_audit)\n    write_json(out / "stage_00_current_pipeline.json", current_pipeline)\n    write_json(out / "stage_01_replay_report_selected.json", replay_report)\n    write_json(out / "stage_02_full_metrics.json", metrics)\n    write_json(out / "stage_03_gap_audit.json", gap)\n    write_json(out / "stage_04_overfit_underfit_audit.json", overfit_underfit)\n    write_json(out / "stage_05_upgrade_stage_reports.json", stage_reports)\n    write_json(out / "stage_06_safe_upgrade_decision.json", decision)\n    claude = make_claude_handoff_text(metrics, gap, source_mode)\n    write_text(out / "CLAUDE_HANDOFF.md", claude)\n\n    # Compact top-level summary.\n    summary = {\n        "version": VERSION,\n        "source_mode": source_mode,\n        "selected_stage": selected_name,\n        "replay_report": replay_report,\n        "MC_certificate_metrics": metrics["groups"]["MC"]["certificate_engine_metrics"],\n        "YNU_certificate_metrics": metrics["groups"]["YNU"]["certificate_engine_metrics"],\n        "ALL_after_safe_override_status_metrics": metrics["groups"]["ALL"]["after_safe_override_status_metrics"],\n        "overfit_underfit_summary": overfit_underfit["summary"],\n    }\n    write_json(out / "SUMMARY.json", summary)\n\n    for p in sorted(out.glob("*")):\n        manifest["files"][p.name] = str(p)\n    write_json(out / "MANIFEST.json", manifest)\n\n    print(json.dumps(summary, indent=2, ensure_ascii=False))\n    print("\\nWrote outputs to:", out)\n    for p in sorted(out.glob("*")):\n        print(" -", p.name)\n    return 0\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
print("Wrote", SCRIPT)


In [ ]:
# Run all stages once. On Kaggle, attach dataset containing exact_eval_round1_Astatine.json first.
OUT = WORK / "exact_v40_5_outputs"
!python {SCRIPT} --out {OUT}


In [ ]:
import json
from pathlib import Path
summary = json.loads((OUT / "SUMMARY.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nOutput files:")
for p in sorted(OUT.glob("*")):
    print("-", p)


## Gate rule

Promote a candidate only if `wrong == []`, `answer_correct == fired`, and `premises_used_correct` does not regress. YNU remains audit-only unless full-log verification passes.
